# Pyr Download Synapse Tables By List

Batch download complete afferent and efferent synapse tables for one or more `zheng_ca3` Pyr/CA3 root IDs, combine each pair with a `direction` column, and save one Parquet plus one metadata JSON per root ID.

## Parameters

In [ ]:
root_ids = [648518346450460332]

datastack_name = "zheng_ca3"
materialization_version = 195
viewer_resolution = [18, 18, 45]

show_full_path = False # When False, project-contained paths display relative to project_root; outside paths remain absolute.
overwrite_existing = False

## CAVE Connection

Load the CAVE token through the shared auth helper without printing it, then connect to the `zheng_ca3` datastack.

In [2]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import caveclient
import pandas as pd

def discover_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers."
    )


project_root = discover_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
synapse_output_dir = project_root / "data" / "synapse_tables"

if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from cave_auth import load_cave_token
from path_behavior import format_path, print_path
import synapse_table_utils as synapse_utils

token, cave_token_source = load_cave_token(project_root)

client = caveclient.CAVEclient(datastack_name, auth_token=token)


## Batch Synapse Table Download

In [3]:
output_dir = Path(synapse_output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

try:
    materialization_timestamp = client.materialize.get_timestamp(materialization_version)
except Exception as exc:
    materialization_timestamp = None
    print(f"Materialization timestamp unavailable: {type(exc).__name__}: {exc}")


In [4]:
batch_start_time = datetime.now(timezone.utc)
batch_results = []

print(f"Batch started: {batch_start_time.isoformat()}")
print(f"root IDs requested: {len(root_ids)}")
print(f"datastack: {datastack_name}")
print(f"materialization_version: {materialization_version}")
print_path("output directory", output_dir, project_root, show_full_path)
print(f"overwrite_existing: {overwrite_existing}")

for root_index, root_id in enumerate(root_ids, start=1):
    root_start_time = datetime.now(timezone.utc)
    print("-" * 80)
    print(f"[{root_index}/{len(root_ids)}] root ID: {root_id}")
    print(f"started: {root_start_time.isoformat()}")

    paths = synapse_utils.artifact_paths(
        output_dir=output_dir,
        root_id=root_id,
        materialization_version=materialization_version,
    )
    parquet_path = paths["parquet_path"]
    metadata_path = paths["metadata_path"]

    if (parquet_path.exists() or metadata_path.exists()) and not overwrite_existing:
        cache_result = synapse_utils.load_valid_synapse_artifact(
            output_dir=output_dir,
            root_id=root_id,
            datastack=datastack_name,
            materialization_version=materialization_version,
            desired_resolution=viewer_resolution,
        )

        root_finish_time = datetime.now(timezone.utc)
        root_duration_seconds = (root_finish_time - root_start_time).total_seconds()

        if cache_result["ok"]:
            metadata = cache_result["metadata"]
            print(f"Valid existing artifact found; skipping CAVE query: {format_path(parquet_path, project_root, show_full_path)}")
            print(f"afferent count: {metadata['afferent_count']}")
            print(f"efferent count: {metadata['efferent_count']}")
            print(f"total synapse rows: {metadata['total_synapse_rows']}")
            status = "cache_valid"
            validation_problems = None
            afferent_count = metadata["afferent_count"]
            efferent_count = metadata["efferent_count"]
            total_synapse_rows = metadata["total_synapse_rows"]
        else:
            print("Existing artifact is missing, invalid, or incompatible; not overwriting because overwrite_existing is False.")
            for problem in cache_result["problems"]:
                print(f"- {problem}")
            status = "cache_invalid"
            validation_problems = "; ".join(cache_result["problems"])
            afferent_count = None
            efferent_count = None
            total_synapse_rows = None

        batch_results.append(
            {
                "root_id": str(root_id),
                "status": status,
                "afferent_count": afferent_count,
                "efferent_count": efferent_count,
                "total_synapse_rows": total_synapse_rows,
                "parquet_path": str(parquet_path),
                "metadata_path": str(metadata_path),
                "validation_problems": validation_problems,
                "started_at": root_start_time.isoformat(),
                "finished_at": root_finish_time.isoformat(),
                "elapsed_seconds": root_duration_seconds,
            }
        )
        continue

    query_result = synapse_utils.query_synapses_for_root(
        client=client,
        root_id=root_id,
        materialization_version=materialization_version,
        desired_resolution=viewer_resolution,
    )
    afferent_df = query_result["afferent_df"]
    efferent_df = query_result["efferent_df"]

    print(f"afferent autapses removed: {query_result['afferent_autapse_count']}")
    print(f"efferent autapses removed: {query_result['efferent_autapse_count']}")
    print(f"afferent count: {len(afferent_df)}")
    print(f"efferent count: {len(efferent_df)}")

    synapses_df = synapse_utils.combine_synapse_tables(afferent_df, efferent_df)

    print(f"combined row count: {len(synapses_df)}")
    print(f"combined DataFrame shape: {synapses_df.shape}")

    metadata = synapse_utils.build_metadata(
        root_id=root_id,
        datastack=datastack_name,
        materialization_version=materialization_version,
        materialization_timestamp=materialization_timestamp,
        desired_resolution=viewer_resolution,
        synapses_df=synapses_df,
        parquet_filename=paths["parquet_filename"],
    )

    save_result = synapse_utils.save_synapse_artifact(
        synapses_df=synapses_df,
        metadata=metadata,
        parquet_path=parquet_path,
        metadata_path=metadata_path,
        overwrite=overwrite_existing,
    )

    if save_result["parquet_written"]:
        print(f"Saved Parquet: {format_path(parquet_path, project_root, show_full_path)}")
    elif save_result["parquet_skipped_existing"]:
        print(f"Parquet file already exists and overwrite_existing is False; leaving unchanged: {format_path(parquet_path, project_root, show_full_path)}")

    if save_result["metadata_written"]:
        print(f"Saved metadata JSON: {format_path(metadata_path, project_root, show_full_path)}")
    elif save_result["metadata_skipped_existing"]:
        print(f"Metadata file already exists and overwrite_existing is False; leaving unchanged: {format_path(metadata_path, project_root, show_full_path)}")

    validation_result = synapse_utils.load_valid_synapse_artifact(
        output_dir=output_dir,
        root_id=root_id,
        datastack=datastack_name,
        materialization_version=materialization_version,
        desired_resolution=viewer_resolution,
    )

    if validation_result["ok"]:
        reloaded_synapses_df = validation_result["synapses_df"]
        print("Parquet and metadata validation passed.")
        print(f"rows: {len(reloaded_synapses_df)}")
        print(f"columns: {len(reloaded_synapses_df.columns)}")
        print(f"direction counts: {reloaded_synapses_df['direction'].value_counts().to_dict()}")
        status = "saved_or_validated"
        validation_problems = None
    else:
        print("Parquet or metadata validation failed:")
        for problem in validation_result["problems"]:
            print(f"- {problem}")
        status = "validation_failed"
        validation_problems = "; ".join(validation_result["problems"])

    root_finish_time = datetime.now(timezone.utc)
    root_duration_seconds = (root_finish_time - root_start_time).total_seconds()
    print(f"finished: {root_finish_time.isoformat()}")
    print(f"elapsed seconds: {root_duration_seconds:.1f}")

    batch_results.append(
        {
            "root_id": str(root_id),
            "status": status,
            "afferent_count": len(afferent_df),
            "efferent_count": len(efferent_df),
            "total_synapse_rows": len(synapses_df),
            "parquet_path": str(parquet_path),
            "metadata_path": str(metadata_path),
            "validation_problems": validation_problems,
            "started_at": root_start_time.isoformat(),
            "finished_at": root_finish_time.isoformat(),
            "elapsed_seconds": root_duration_seconds,
        }
    )

batch_finish_time = datetime.now(timezone.utc)
batch_duration_seconds = (batch_finish_time - batch_start_time).total_seconds()
print("-" * 80)
print(f"Batch finished: {batch_finish_time.isoformat()}")
print(f"Batch elapsed seconds: {batch_duration_seconds:.1f}")


Batch started: 2026-08-29T15:26:26.098805+00:00
root IDs requested: 1
datastack: zheng_ca3
materialization_version: 195
output directory: data\synapse_tables
overwrite_existing: False
--------------------------------------------------------------------------------
[1/1] root ID: 648518346450460332
started: 2026-08-29T15:26:26.098805+00:00
afferent autapses removed: 80
efferent autapses removed: 80
afferent count: 4807
efferent count: 29
combined row count: 4836
combined DataFrame shape: (4836, 13)
Saved Parquet: data\synapse_tables\synapses_648518346450460332_mat195.parquet
Saved metadata JSON: data\synapse_tables\synapses_648518346450460332_mat195_metadata.json
Parquet and metadata validation passed.
rows: 4836
columns: 13
direction counts: {'afferent': 4807, 'efferent': 29}
finished: 2026-08-29T15:26:34.693078+00:00
elapsed seconds: 8.6
--------------------------------------------------------------------------------
Batch finished: 2026-08-29T15:26:34.693078+00:00
Batch elapsed secon

## Batch Summary

In [5]:
batch_results_df = pd.DataFrame(batch_results)
batch_results_display_df = batch_results_df.copy()
for path_column in ["parquet_path", "metadata_path"]:
    if path_column in batch_results_display_df.columns:
        batch_results_display_df[path_column] = batch_results_display_df[path_column].map(
            lambda path: format_path(path, project_root, show_full_path)
        )
display(batch_results_display_df)

,root_id,status,afferent_count,efferent_count,total_synapse_rows,parquet_path,metadata_path,validation_problems,started_at,finished_at,elapsed_seconds
0,648518346450460332,saved_or_validated,4807,29,4836,data\synapse_tables\synapses_64851834645046033...,data\synapse_tables\synapses_64851834645046033...,None,2026-08-29T15:26:26.098805+00:00,2026-08-29T15:26:34.693078+00:00,8.594273
